# CNN Model from Scratch

This notebook builds and trains a Convolutional Neural Network from scratch for pneumonia detection.

In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from src.models.cnn_scratch import build_cnn_model
from src.models.train import train_model
from src.models.evaluate import evaluate_model, get_confusion_matrix, get_classification_report
from src.utils.visualization import plot_training_history, plot_confusion_matrix, plot_roc_curve
from src.utils.config import CLASS_LABELS, BEST_MODEL_PATH, IMG_SIZE, NUM_CLASSES

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 1. Load Preprocessed Data

Note: Run the preprocessing notebook first to generate the preprocessed data.

In [ ]:
# Load preprocessed data (assuming it was saved in preprocessing notebook)
# If not saved, you'll need to run preprocessing steps here
# For now, we'll assume data is available from previous notebook execution

# Uncomment if you saved preprocessed data:
# import pickle
# with open('data/processed/preprocessed_data.pkl', 'rb') as f:
#     data = pickle.load(f)
#     X_train = data['X_train']
#     X_val = data['X_val']
#     X_test = data['X_test']
#     y_train = data['y_train']
#     y_val = data['y_val']
#     y_test = data['y_test']

# Otherwise, load and preprocess here (see preprocessing notebook)
print("Please ensure preprocessed data is loaded (X_train, X_val, X_test, y_train, y_val, y_test)")

## 2. Build CNN Model

In [ ]:
# Build CNN model
input_shape = (*IMG_SIZE, 1)  # Grayscale images
model = build_cnn_model(input_shape=input_shape, num_classes=NUM_CLASSES)

# Display model architecture
model.summary()

## 3. Train the Model

In [ ]:
# Train model
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    epochs=50,
    batch_size=32,
    use_class_weights=True,
    use_augmentation=True,
    model_save_path=BEST_MODEL_PATH,
    logs_dir='logs/training_logs'
)

## 4. Plot Training History

In [ ]:
# Plot training history
plot_training_history(history)

## 5. Evaluate Model Performance

In [ ]:
# Load best model
best_model = keras.models.load_model(BEST_MODEL_PATH)

# Evaluate on test set
metrics = evaluate_model(best_model, X_test, y_test)

print("Test Set Performance:")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1-Score: {metrics['f1_score']:.4f}")
print(f"ROC-AUC: {metrics['roc_auc']:.4f}")

In [ ]:
# Confusion Matrix
cm = get_confusion_matrix(best_model, X_test, y_test)
plot_confusion_matrix(y_test, 
                     np.argmax(best_model.predict(X_test, verbose=0), axis=1),
                     class_names=list(CLASS_LABELS.values()))

In [ ]:
# Classification Report
report = get_classification_report(best_model, X_test, y_test, 
                                  target_names=list(CLASS_LABELS.values()))
print(report)

In [ ]:
# ROC Curve
y_pred_proba = best_model.predict(X_test, verbose=0)
plot_roc_curve(y_test, y_pred_proba, class_names=list(CLASS_LABELS.values()))

## 6. Model Performance Comments

### Observations:
- **Training Progress**: The model shows [describe training behavior]
- **Validation Performance**: [describe validation metrics]
- **Test Performance**: [describe test metrics]
- **Overfitting**: [check for overfitting signs]
- **Class Performance**: [analyze per-class performance]

### Recommendations:
- [Suggest improvements based on results]